# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LeylaAghayeva1/ml-search-engineering/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method choice and why

I chose Logistic Regression because this task is a binary classification problem: predicting whether a content page is declining and may require a refresh.

The model fits this lane because it produces probability scores that can be used to rank pages by refresh priority. This matches the Week-4 baseline objective of creating a ranked refresh queue.

Logistic Regression was selected because it is simple, interpretable, and provides a strong first comparison against the rule-based baseline. It allows us to understand which observed signals contribute to the prediction before introducing more complex models.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


df = pd.read_csv(
    "../../data/raw/content_refresh_anonymized.csv"
)


# Create target label
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)


features = [
    "search_volume",
    "competition",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position"
]


X = df[features].copy()

y = df["is_declining_label"]

groups = df["client_id"]


print("Features:", X.shape)
print("Label distribution:")
print(y.value_counts())

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split design

I used a grouped split based on client_id.

This split is more honest because multiple pages can belong to the same client. If content from the same client appeared in both training and testing data, the model could learn client-specific patterns instead of learning general refresh signals.

Grouping by client_id helps evaluate whether the model can generalize to content from unseen client groups and reduces the risk of overly optimistic evaluation results.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)


train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)


X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]


y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]


print("Training rows:", X_train.shape)
print("Testing rows:", X_test.shape)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train + compare vs baseline

The Logistic Regression model was evaluated using the same prediction objective and ranking metrics as the Week-4 baseline.

The Week-4 baseline created a refresh priority score using transparent rules based on content freshness, search demand, and visibility. The Logistic Regression model instead learned patterns from the available features and produced probability scores that were used to rank refresh candidates.

The model evaluation used a grouped client split to provide an honest validation design, while the baseline score was recalculated using the same dataset for comparison.

The comparison uses Precision@20 and Precision@50 because the goal is to identify high-quality refresh candidates near the top of the ranked queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.impute import SimpleImputer


model = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median"
        )
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])


model.fit(
    X_train,
    y_train
)



In [ ]:
results = X_test.copy()

results["actual"] = y_test

results["model_score"] = (
    model.predict_proba(X_test)[:,1]
)

results.head()

In [ ]:
model_p20 = precision_at_k(
    results,
    "model_score",
    "actual",
    20
)

model_p50 = precision_at_k(
    results,
    "model_score",
    "actual",
    50
)

print("Model Precision@20:", model_p20)
print("Model Precision@50:", model_p50)

In [ ]:
comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Logistic Regression"
    ],
    "Precision@20": [
        0.50,
        model_p20
    ],
    "Precision@50": [
        0.46,
        model_p50
    ]
})


comparison

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and interpretation

The model produced 2,832 errors on the test set. These errors were reviewed to understand where the model struggles and which signals influence its decisions.

The strongest positive signals used by the model were days_since_last_update, word_count, and impressions_90d. The relationship between days_since_last_update and declining content is directionally reasonable because stale pages may require review.

Some feature relationships were not always intuitive. For example, content_age_days had a negative coefficient, showing that the model learned a relationship from the observed dataset rather than simply assuming that older pages always decline. This relationship should be investigated further before using the feature for automated decisions.

The main error categories were:
- False negatives: pages that were actually declining but received lower model scores. These pages may still have strong visibility signals that make them appear healthier.
- False positives: pages predicted as declining but not labeled as declining. These pages may share similar characteristics with declining content.

The Logistic Regression model improved Precision@20 and Precision@50 compared with the Week-4 baseline. However, the model should be treated as decision support rather than an automatic refresh decision system.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
importance = pd.DataFrame({
    "feature": features,
    "weight": model.named_steps["model"].coef_[0]
})


importance = importance.sort_values(
    "weight",
    ascending=False
)


importance

In [ ]:
# Create binary predictions for error analysis
results["prediction"] = (
    results["model_score"] >= 0.5
).astype(int)


# Find incorrect predictions
errors = results[
    results["prediction"] != results["actual"]
]


print("Total errors:", len(errors))


# Show examples with scores and important signals
errors[
    [
        "actual",
        "prediction",
        "model_score",
        "days_since_last_update",
        "content_age_days",
        "impressions_90d",
        "ctr"
    ]
].head(10)

In [ ]:
importance = pd.DataFrame({
    "feature": features,
    "weight": model.named_steps["model"].coef_[0]
})


importance.sort_values(
    "weight",
    ascending=False
)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.